# Etap preprocessingu - krok 2: konwersja czasu UTC -> Europe/Warsaw

Cel: przeliczyć znaczniki czasu (epoch, UTC) na czas lokalny, z poprawnym
zachowaniem przy nocy zmiany czasu (25.10.2026 w próbce).

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from config import RAW, STATIC
from src.analysis.consolidate import consolidate_trip_updates
from src.analysis.schedule import fill_stop_sequence
from src.analysis.localtime import to_local_time, add_local_columns

consolidated = consolidate_trip_updates(RAW, feed="A")
filled = fill_stop_sequence(consolidated, STATIC, feed="A")
print("Liczba wierszy:", len(filled))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Liczba wierszy: 691933


## Test syntetyczny — zachowanie przy nocy zmiany czasu

Sprawdzamy wprost, czy dwa momenty UTC oddalone o dokładnie godzinę, po obu stronach
momentu zmiany czasu (25.10.2026, 01:00 UTC), powinny dać **tę samą godzinę
lokalną**, bo lokalna godzina 02:00-03:00 wystąpiła tej nocy dwukrotnie.

In [2]:
moment_przed = pd.Timestamp("2026-10-25 00:30:00", tz="UTC")
moment_po = moment_przed + pd.Timedelta(hours=1)

epoch_przed = int(moment_przed.timestamp())
epoch_po = int(moment_po.timestamp())

test = pd.Series([epoch_przed, epoch_po])
lokalny = to_local_time(test)

print("UTC:", moment_przed, "->", lokalny.iloc[0])
print("UTC:", moment_po, "->", lokalny.iloc[1])
print("\nTa sama godzina lokalna mimo roznicy 1h w UTC:",
      lokalny.iloc[0].hour == lokalny.iloc[1].hour)

UTC: 2026-10-25 00:30:00+00:00 -> 2026-10-25 02:30:00+02:00
UTC: 2026-10-25 01:30:00+00:00 -> 2026-10-25 02:30:00+01:00

Ta sama godzina lokalna mimo roznicy 1h w UTC: True


## Zastosowanie na danych rzeczywistych

In [3]:
with_local = add_local_columns(filled, epoch_column="arrival_time", prefix="arrival")
with_local = add_local_columns(with_local, epoch_column="last_observed_at", prefix="observed")

print(with_local[["date", "arrival_time", "arrival_local", "arrival_date_local",
                  "arrival_hour_local", "arrival_weekday_local"]].head())

        date  arrival_time             arrival_local arrival_date_local  \
0 2026-10-21  1.792621e+09 2026-10-22 00:21:43+02:00         2026-10-22   
1 2026-10-21  1.792622e+09 2026-10-22 00:29:00+02:00         2026-10-22   
2 2026-10-21  1.792621e+09 2026-10-22 00:23:35+02:00         2026-10-22   
3 2026-10-21  1.792622e+09 2026-10-22 00:34:03+02:00         2026-10-22   
4 2026-10-21  1.792622e+09 2026-10-22 00:38:38+02:00         2026-10-22   

   arrival_hour_local  arrival_weekday_local  
0                 0.0                    3.0  
1                 0.0                    3.0  
2                 0.0                    3.0  
3                 0.0                    3.0  
4                 0.0                    3.0  


In [4]:
noc_zmiany = with_local[with_local["arrival_date_local"] == pd.Timestamp("2026-10-25").date()]
rozklad_godzin = noc_zmiany["arrival_hour_local"].value_counts().sort_index()
print(rozklad_godzin)

arrival_hour_local
0.0      989
1.0      687
2.0     1287
3.0     1019
4.0     2218
5.0     4653
6.0     5348
7.0     5540
8.0     5903
9.0     6198
10.0    6875
11.0    6795
12.0    7075
13.0    6846
14.0    6802
15.0    7001
16.0    6945
17.0    6753
18.0    6438
19.0    5810
20.0    5878
21.0    5594
22.0    5067
23.0    3042
Name: count, dtype: int64


In [5]:
TARGET_DATES = ["2026-10-22", "2026-10-23", "2026-10-25", "2026-10-26"]

# Po dodaniu godzin z 21.10/24.10, luki w lokalnych godzinach 0-1 dla dni 22 i 25
# powinny zniknąc. Weryfikacja:
for d in TARGET_DATES:
    day_df = with_local[with_local["arrival_date_local"] == pd.Timestamp(d).date()]
    godziny = sorted(day_df["arrival_hour_local"].dropna().unique())
    print(d, "-> najwczesniejsza godzina:", godziny[0] if godziny else "brak", "| liczba godzin:", len(godziny))

# Zakres poza TARGET_DATES (np. "widmowy" 24.10) to oczekiwany efekt obciecia
# probki na granicach dni UTC -- nie bedacy czescia analizy, odcinamy swiadomie:
w_zakresie = with_local[with_local["arrival_date_local"].astype(str).isin(TARGET_DATES)]
print("\nWiersze w zakresie:", len(w_zakresie), "/ wszystkie:", len(with_local))

2026-10-22 -> najwczesniejsza godzina: 0.0 | liczba godzin: 24
2026-10-23 -> najwczesniejsza godzina: 0.0 | liczba godzin: 24
2026-10-25 -> najwczesniejsza godzina: 0.0 | liczba godzin: 24
2026-10-26 -> najwczesniejsza godzina: 0.0 | liczba godzin: 20

Wiersze w zakresie: 658876 / wszystkie: 691933


In [6]:
local_24 = with_local[with_local["arrival_date_local"] == pd.Timestamp("2026-10-24").date()]
print("Liczba wierszy z arrival_date_local = 2026-10-24:", len(local_24))
print("\nZ jakiej partycji (date) pochodza:")
print(local_24["date"].value_counts())
print("\nIle z nich ma flage uncertain_completion:")
print(local_24["uncertain_completion"].value_counts())
print("\nPrzyklad:")
print(local_24[["date", "trip_id", "arrival_time", "last_observed_at", "uncertain_completion"]].head(10))

Liczba wierszy z arrival_date_local = 2026-10-24: 3888

Z jakiej partycji (date) pochodza:
date
2026-10-23    3888
Name: count, dtype: int64

Ile z nich ma flage uncertain_completion:
uncertain_completion
False    3888
Name: count, dtype: Int64

Przyklad:
             date                  trip_id  arrival_time  last_observed_at  \
213199 2026-10-23   20261021_4_129511561_8  1.792794e+09        1792792864   
213200 2026-10-23   20261021_4_129511562_9  1.792801e+09        1792792374   
213201 2026-10-23   20261021_4_129511562_9  1.792801e+09        1792792374   
213202 2026-10-23   20261021_4_129511562_9  1.792801e+09        1792792374   
213203 2026-10-23  20261021_4_129507880_16  1.792793e+09        1792792864   
213204 2026-10-23   20261024_4_129739771_5  1.792793e+09        1792793445   
213205 2026-10-23   20261024_4_129739782_7  1.792794e+09        1792794370   
213206 2026-10-23  20261024_4_129737938_23  1.792794e+09        1792793606   
213207 2026-10-23   20261024_4_129739820_3

In [7]:
cross = (
    with_local
    .groupby(["date", "arrival_date_local"])
    .size()
    .reset_index(name="liczba_wierszy")
    .sort_values(["arrival_date_local", "date"])
)
print(cross.to_string(index=False))

      date arrival_date_local  liczba_wierszy
2026-10-22         2026-10-22          199086
2026-10-23         2026-10-22              49
2026-10-22         2026-10-23            3016
2026-10-23         2026-10-23          202728
2026-10-23         2026-10-24            3888
2026-10-25         2026-10-25          118128
2026-10-26         2026-10-25             310
2026-10-25         2026-10-26            1620
2026-10-26         2026-10-26          129067


## Wnioski

1. **Mechanizm konwersji działa poprawnie**: Kierunek epoch/UTC -> czas lokalny jest zawsze
   jednoznaczny. Potwierdzone jest to testem syntetycznym, czyli dwa momenty UTC oddalone o godzinę, po obu stronach zmiany czasu
   25.10.2026, dają tę samą godzinę lokalną (02:30).

2. **Efekt podwójnej godziny lokalnej**: W noc zmiany czasu (25.10.2026) godzina lokalna
   02:00–03:00 wystąpiła dwukrotnie w UTC. Przy agregacji po samej godzinie lokalnej
   ten fragment doby zawiera dane z dwóch różnych godzin UTC.
   Trzeba o tym pamiętać przy interpretacji wyników nocnych z tego dnia.

3. **Odkryto i naprawiono lukę na granicach próbki**: Dla dni 22.10 i 25.10 brakowało
   lokalnych godzin 0–1, ponieważ te godziny fizycznie leżą w pliku UTC *poprzedniego* dnia
   (Polska to UTC+1/+2, więc północ lokalna wypada 1–2h przed północą UTC), a próbka nie
   zawierała dni 21.10 i 24.10. Naprawione to zostało przez dociągnięcie z backupu tylko brakujących
   dwóch godzin z każdego z tych dni. Po naprawie wszystkie cztery docelowe dni mają komplet godzin lokalnych.

4. **Odkryto odrębny efekt brzegowy próbki**: Obserwacje z końca partycji 23.10 UTC zawierają świeże,
   jeszcze niezrealizowane prognozy (`uncertain_completion = False`) dla kursów nocnych, których `arrival_time` wypada lokalnie
   już 24.10, czyli dnia spoza zamierzonego zakresu próbki. Jest to naturalna konsekwencja wycinania pojedynczych
   dni UTC z czegoś ciągłego w czasie lokalnym. Dociąganie kolejnych godzin tylko przesunęłoby problem na następny dzień.
   Rozwiązanie: jawne ograniczenie analizy do czterech docelowych dni (`TARGET_DATES`), zamiast prób
   złapania granicy.

5. **Finalny, przefiltrowany zbiór do dalszej pracy ma 658 876 wierszy**: Różnica względem wcześniejszej
   liczby skonsolidowanych par wynika z trzech przyczyn: wierszy bez `arrival_time`, efektu brzegowego z punktu 4
   oraz nowych partycji 21.10 i 24.10 dodanych tylko we fragmentach (2h), które same w sobie leżą
   poza docelowym zakresem dat.

6. **Zaimplementowano jako funkcje dodające kolumny** (`arrival_local`, `arrival_date_local`,
   `arrival_hour_local`, `arrival_weekday_local`), nie nadpisujące oryginalnej epoki.
   Jest zachowana możliwość powrotu do źródła przy debugowaniu.